# Write RDF from a chemical table

Use the generated Chemical class to turn names and IRIs into RDF. The model supplies the class and property IRIs, so the table conversion does not need to repeat them.

Run the [thyroid investigation](AOPWiki_hydration.ipynb) first. We use its real chemical table and saved schema; this notebook makes no endpoint requests.

In [1]:
import json
from pathlib import Path

import pandas as pd
from rdflib import Graph
from rdfsolve import MinedSchema

session = json.loads(Path("thyroid-session.json").read_text())
schema = MinedSchema.from_dict(session["schema"])
Chemical = schema.to_pydantic_classes()["ChemicalEntity"]
table = pd.read_csv("thyroid-chemicals.csv")
table[["uri", "title"]]

,uri,title
0,https://identifiers.org/cas/115-86-6,Triphenyl phosphate
1,https://identifiers.org/cas/117718-60-2,Thiazopyr
2,https://identifiers.org/cas/131-55-5,"2,2',4,4'-Tetrahydroxybenzophenone"
3,https://identifiers.org/cas/17737-65-4,Clonixin
4,https://identifiers.org/cas/50-06-6,Phenobarbital
5,https://identifiers.org/cas/51-52-5,6-Propyl-2-thiouracil
6,https://identifiers.org/cas/55335-06-3,Triclopyr
7,https://identifiers.org/cas/60-56-0,Methimazole
8,https://identifiers.org/cas/644-62-2,Meclofenamic acid
9,https://identifiers.org/cas/79-94-7,"3,3',5,5'-Tetrabromobisphenol A"


## Create typed records

Each row states that the IRI is a chemical with this title. We intentionally import only these two columns. We are creating a small set of assertions, not reconstructing all annotations from the source.

In [2]:
records = [Chemical(uri=row.uri, title=row.title) for row in table.itertuples()]
records[0]

ChemicalEntity(uri='https://identifiers.org/cas/115-86-6', rdf_type=[], identifier=None, source=None, title='Triphenyl phosphate', alternative=None, ispartof=None, cheminf_000059=None, cheminf_000446=None, cheminf_000568=None, type=None, label=None, sameas=None, exactmatch=None)

## Save RDF

`to_graph()` uses the field's RDF definition to distinguish a literal title from a resource reference. It raises if that definition leaves the RDF term kind or datatype ambiguous.

In [3]:
graph = Graph()
for record in records:
    graph += record.to_graph()

graph.serialize("chemicals-from-table.ttl", format="turtle")
print(records[0].to_graph().serialize(format="turtle"))

@prefix dc: <http://purl.org/dc/elements/1.1/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://identifiers.org/cas/115-86-6> a <http://semanticscience.org/resource/CHEMINF_000000> ;
    dc:title "Triphenyl phosphate"^^xsd:string .




## Read it through the same API

The result is ordinary RDF. It can be read by RDF tools or by rdfsolve's generated classes.

In [4]:
saved = Graph().parse("chemicals-from-table.ttl", format="turtle")
with schema.client(saved, graph_uris=[]) as client:
    found = client.search(client.model("ChemicalEntity"), "Phenobarbital", fields=["title"])
    display(client.table(found, ["title"]))

assert len(records) == len(table)
assert set(saved) == set(graph)

,uri,title
0,https://identifiers.org/cas/50-06-6,Phenobarbital


A plain table does not retain the source's language tags, literal spelling, or provenance. Use retrieved objects and their `rdf_terms` when those details must survive. For new ambiguous values, supply explicit RDF terms rather than guessing.

Pydantic validates the Python fields; exporting them does not run SHACL validation. The model's class is asserted for a newly created record. No mapping adds further types or changes the record's IRI.